In [1]:
# Import necessary libraries
import numpy as np
import pandas as pd
import os

# Set seed index for model reproducibility and folder naming
seed_index = 50

# Define folder path for storing model outputs based on seed index
folder_path = "/Net/Groups/BGI/scratch/fhuang/sm_utb_cnn/model/c1d-2/" + str(seed_index)

# Check if the directory exists, and create it if it doesn't
# This ensures the output folder is available for saving model results
if not os.path.exists(folder_path):  # Check if folder exists, create if not exists
    os.makedirs(folder_path)  # Create directory recursively

# 1.load data and split data

In [2]:
# Load input data arrays
s1 = np.load('/Net/Groups/BGI/scratch/fhuang/sm_uta_cnn/input_data/sa/idata.npy')
s2 = np.load('/Net/Groups/BGI/scratch/fhuang/sm_utb_cnn/input/sm_somo.npy')

# Load indices of NaN values to be excluded
nan_idx = np.load('/Net/Groups/BGI/scratch/fhuang/sm_uta_cnn/conv3d_res/nan_idx.npy')

# Combine data from both sources, replacing the last dimension of s1 with s2's last dimension
xy = s1
xy[:, :, -1] = s2[:, :, -1]  # Replace last feature with soil moisture data from s2

# Remove samples containing NaN values
xy = np.delete(xy, nan_idx, axis=0)

# Get total number of samples after NaN removal
len_data = xy.shape[0]

# Create shuffled indices for random train/validation/test split
shuffled_indices = np.random.permutation(len_data)

# Split indices into training (70%), validation (10%), and test (20%) sets
train_idx = shuffled_indices[:int(0.7 * len_data)]
val_idx = shuffled_indices[int(0.7 * len_data):int(0.8 * len_data)]
test_idx = shuffled_indices[int(0.8 * len_data):]

# Prepare training data: reshape from [samples, time, features] to [samples*time, features]
# Features 4-11 (8 features) used as input, feature 13 as target
x_train = xy[train_idx, :, 4:12].reshape(train_idx.shape[0] * 216, 8)
y_train = xy[train_idx, :, 13:].reshape(train_idx.shape[0] * 216, 1)

# Prepare validation data with same reshaping
x_val = xy[val_idx, :, 4:12].reshape(val_idx.shape[0] * 216, 8)
y_val = xy[val_idx, :, 13:].reshape(val_idx.shape[0] * 216, 1)

# Prepare test data with same reshaping
x_test = xy[test_idx, :, 4:12].reshape(test_idx.shape[0] * 216, 8)
y_test = xy[test_idx, :, 13:].reshape(test_idx.shape[0] * 216, 1)

# Prepare complete dataset (all samples) for potential use
# First 12 features (including coordinates: lati, loni, lat, lon) and target variable
x_xy = xy[:, :, :12].reshape(len_data * 216, 12)  # First 4 are lati, loni, lat, lon
y_xy = xy[:, :, 13:].reshape(len_data * 216, 1)

# Print shapes to verify data dimensions
print(x_test.shape, x_val.shape, y_test.shape, x_train.shape)
print(x_xy.shape, y_xy.shape)

(31968, 8) (15984, 8) (31968, 1) (111240, 8)
(159192, 12) (159192, 1)


In [3]:
import pandas as pd

# Convert numpy arrays to pandas DataFrames for easier data manipulation
df_train = pd.DataFrame(x_train)
df_train["y"] = y_train  # Add target variable column
df_train = df_train.fillna(0)  # Fill any NaN values with 0

df_val = pd.DataFrame(x_val)
df_val["y"] = y_val
df_val = df_val.fillna(0)

df_test = pd.DataFrame(x_test)
df_test["y"] = y_test
df_test = df_test.fillna(0)

# Create DataFrame for complete dataset (all samples)
df_xy = pd.DataFrame(x_xy)
df_xy["y"] = y_xy
df_xy_no_Nan = df_xy.fillna(0)  # Remove NaN values

# Initialize DataFrames for min-max normalized data (same structure as originals)
df_train_maxmin = df_train * 0
df_val_maxmin = df_val * 0
df_test_maxmin = df_test * 0

# For complete dataset, only normalize features 4+ (excluding coordinate columns 0-3)
df_xy_maxmin = df_xy_no_Nan.iloc[:, 4:] * 0

# Apply min-max normalization to each feature (0-8 range, 9 features total including target)
for i in range(9):
    # Calculate min and max values from training data only (to avoid data leakage)
    maxv = df_train.iloc[:, i].max()
    minv = df_train.iloc[:, i].min()
    
    # Apply min-max normalization: (value - min) / (max - min)
    df_train_maxmin.iloc[:, i] = (df_train.iloc[:, i] - minv) / (maxv - minv)
    df_val_maxmin.iloc[:, i] = (df_val.iloc[:, i] - minv) / (maxv - minv)
    df_test_maxmin.iloc[:, i] = (df_test.iloc[:, i] - minv) / (maxv - minv)
    
    # For complete dataset, normalize only features starting from column 4
    # This excludes coordinate columns (lati, loni, lat, lon) from normalization
    df_xy_maxmin.iloc[:, i] = (df_xy_no_Nan.iloc[:, 4 + i] - minv) / (maxv - minv)

# 2. local-only model structure

In [5]:
# Import PyTorch and related libraries for deep learning
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.nn import L1Loss
from ignite.contrib.metrics.regression.r2_score import R2Score  # For R-squared metric
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn import Conv1d, Conv2d, Conv3d
from torch.nn import MaxPool1d, MaxPool2d, MaxPool3d
from torch.nn import Flatten
from torch.nn import Linear
from torch.nn.functional import relu
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import SGD
from torch.optim import Adam
from torch.nn import L1Loss
from torch import nn
from ignite.contrib.metrics.regression.r2_score import R2Score
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

# Define 1D CNN Regressor model for time series regression tasks
class CnnRegressor(torch.nn.Module):
    def __init__(self, batch_size, inputs, outputs):
        super(CnnRegressor, self).__init__()
        # Initialize model parameters
        self.batch_size = batch_size
        self.inputs = inputs  # Number of input features
        self.outputs = outputs  # Number of output predictions
        
        # Define model architecture layers
        self.input_layer = Conv1d(inputs, batch_size, 1, stride=1)  # Initial 1D convolution
        self.max_pooling_layer = MaxPool1d(1)  # Max pooling with kernel size 1
        self.conv_layer1 = Conv1d(batch_size, 256, 1, stride=3)  # First convolutional block
        self.conv_layer2 = Conv1d(256, 512, 1, stride=3)  # Second convolutional block
        self.conv_layer3 = Conv1d(512, 512, 1, stride=3)  # Third convolutional block
        self.flatten_layer = Flatten()  # Flatten layer for transition to dense layers
        self.linear_layer = Linear(512, 128)  # First fully connected layer
        self.output_layer = Linear(128, outputs)  # Final output layer

    def forward(self, input):
        # Reshape input to [batch_size, input_features, sequence_length=1]
        input = input.reshape((self.batch_size, self.inputs, 1))
        
        # Forward pass through the network
        output = relu(self.input_layer(input))  # Input convolution with ReLU activation
        output = self.max_pooling_layer(output)  # Apply max pooling
        
        output = relu(self.conv_layer1(output))  # First conv block with ReLU
        output = self.max_pooling_layer(output)  # Max pooling
        
        output = relu(self.conv_layer2(output))  # Second conv block with ReLU
        output = self.max_pooling_layer(output)  # Max pooling
        
        output = relu(self.conv_layer3(output))  # Third conv block with ReLU
        output = self.flatten_layer(output)  # Flatten for dense layers
        
        output = relu(self.linear_layer(output))  # Fully connected layer with ReLU
        output = self.output_layer(output)  # Final output layer (no activation)
        
        return output

/tmp/ipykernel_2434710/1572771537.py:6: DeprecationWarning: /Net/Groups/BGI/scratch/fhuang/.conda/envs/hfng/lib/python3.10/site-packages/ignite/contrib/metrics/regression/r2_score.py has been moved to ignite/metrics/regression/r2_score.py and will be removed in version 0.6.0
  from ignite.contrib.metrics.regression.r2_score import R2Score


# 3. model training

In [6]:
# Prepare training data tensors
# Convert normalized training features to tensor and add channel dimension for Conv1D
train_inputs = torch.from_numpy(np.expand_dims(df_train_maxmin.iloc[:,:-1].values, axis=-1)).cuda().float()
# Convert normalized training target values to tensor and reshape to [samples, 1]
train_outputs = torch.from_numpy(df_train_maxmin.iloc[:,-1].values.reshape(
    df_train_maxmin.iloc[:,-1].values.shape[0], 1)).cuda().float()
# Create TensorDataset for training data (input-output pairs)
train_tensor = TensorDataset(train_inputs, train_outputs)
# Create DataLoader for training with batching and shuffling
train_loader = DataLoader(train_tensor, batch_size=100, shuffle=True, drop_last=True)

# Prepare validation data tensors
# Convert normalized validation features to tensor with channel dimension
val_inputs = torch.from_numpy(np.expand_dims(df_val_maxmin.iloc[:,:-1].values, axis=-1)).cuda().float()
# Convert normalized validation target values to tensor
val_outputs = torch.from_numpy(df_val_maxmin.iloc[:,-1].values.reshape(
    df_val_maxmin.iloc[:,-1].values.shape[0], 1)).cuda().float()
# Create TensorDataset for validation data
val_tensor = TensorDataset(val_inputs, val_outputs)
# Create DataLoader for validation with same batch size
val_loader = DataLoader(val_tensor, batch_size=100, shuffle=True, drop_last=True)

# Prepare test data tensors
# Convert normalized test features to tensor with channel dimension
test_inputs = torch.from_numpy(np.expand_dims(df_test_maxmin.iloc[:,:-1].values, axis=-1)).cuda().float()
# Convert normalized test target values to tensor
test_outputs = torch.from_numpy(df_test_maxmin.iloc[:,-1].values.reshape(
    df_test_maxmin.iloc[:,-1].values.shape[0], 1)).cuda().float()
# Create TensorDataset for test data
test_tensor = TensorDataset(test_inputs, test_outputs)
# Create DataLoader for testing
test_loader = DataLoader(test_tensor, batch_size=100, shuffle=True, drop_last=True)

In [7]:
y1_max=0.1609139
y1_min=-0.11246508

In [1]:
# Set device to GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 100

# Initialize CNN regression model
model = CnnRegressor(batch_size, 8, 1)

# Multi-GPU setup (commented out)
# if torch.cuda.device_count() > 1:
#      print("Let's use", torch.cuda.device_count(), "GPUs!")
#      model = torch.nn.DataParallel(model,device_ids=[0, 2, 3, 4, 5, 6, 7]).to(device)

# Move model to specified device (GPU/CPU)
model.to(device)

# Weight initialization (commented out)
# model.apply(weigth_init)

# Define loss function (Mean Squared Error)
loss_fn = nn.MSELoss()

# Define optimizer (Adam)
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Define normalization parameters for output scaling
y1_min=-0.112465
y1_max= 0.1609139

# Training parameters
n_epochs = 40  # Total number of training epochs

# Lists to track training and validation loss
train_loss_list=[]
val_loss_list=[]

# Main training loop
for epoch in range(n_epochs):
    # Iterate through training batches
    for batch_x, batch_y in train_loader:
        # Reset gradients
        optimizer.zero_grad()
        
        # Move batch data to device
        batch_x=batch_x.to(device)
        
        # Denormalize target values
        batch_y=((batch_y)*(y1_max-y1_min)+y1_min).to(device)
        
        # Forward pass: compute predictions
        y1_pred= model(batch_x)
        
        # Denormalize predictions
        y1_pred=((y1_pred)*(y1_max-y1_min)+y1_min).to(device)
        
        # Compute loss between predictions and targets
        loss = loss_fn(y1_pred, batch_y)
        
        # Backward pass: compute gradients
        loss.mean().backward()
        
        # Update model parameters
        optimizer.step()

    # Validation phase (no gradient computation)
    with torch.no_grad():
        val_loss = 0
        # Iterate through validation batches
        for val_batch_x, val_batch_y in val_loader:
            val_batch_x=val_batch_x.to(device)
            
            # Denormalize validation targets
            val_batch_y=((val_batch_y)*(y1_max-y1_min)+y1_min).to(device)
            
            # Forward pass on validation data
            y1_val_pred= model(val_batch_x.cuda())  
            
            # Denormalize validation predictions
            y1_val_pred=((y1_val_pred)*(y1_max-y1_min)+y1_min).to(device)
            
            # Compute validation loss
            val_loss = loss_fn(y1_val_pred, val_batch_y)
            val_loss += val_loss

        # Calculate average validation loss
        val_loss /= len(val_loader)

    # Print training progress
    print(f"Epoch {epoch+1}/{n_epochs} - Train Loss: {loss.item():.4f} - Val Loss: {val_loss.item():.4f}")
    
    # Store losses for tracking
    train_loss_list.append(loss.item())
    val_loss_list.append(val_loss.item())

# 4. model testing

In [10]:
# Initialize lists to store test predictions and observations
y_test_pred_list=[]
y_test_obs_list=[]

# Iterate through test data batches
for test_batch_x, test_batch_y in test_loader:
    # Move test input data to device (GPU/CPU)
    test_batch_x=test_batch_x.to(device)
    
    # Denormalize test target values using predefined min/max
    test_batch_y=(test_batch_y*(y1_max-y1_min)+y1_min).to(device)
    
    # Forward pass: get model predictions
    y_test_pred = model(test_batch_x)
    
    # Denormalize predictions to original scale
    y1_test_pred=((y_test_pred)*(y1_max-y1_min)+y1_min)
    
    # Store predictions and observations
    y_test_pred_list.append(y1_test_pred)
    y_test_obs_list.append(test_batch_y)

# Concatenate all batch results into single tensors
y_test_pred_np=torch.cat(y_test_pred_list)
y_test_obs_np=torch.cat(y_test_obs_list)

# Import evaluation metrics
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error
import pandas as pd

# Convert tensors to numpy arrays and reshape for analysis
y_pred=y_test_pred_np.cpu().detach().numpy().reshape(-1,1)[:,0]  # Move to CPU, detach from graph, convert to numpy
y_obs= y_test_obs_np.cpu().detach().numpy().reshape(-1,1)[:,0]   # Move to CPU, detach from graph, convert to numpy

# Create DataFrame for easy comparison and handle missing values
df_pred_obs=pd.DataFrame({"pred":y_pred,"obs":y_obs}).dropna(axis=0,how="any")

# Calculate R-squared (coefficient of determination)
r1=r2_score(df_pred_obs["obs"],df_pred_obs["pred"])

# Calculate Mean Squared Error
mse=mean_squared_error(df_pred_obs["obs"],df_pred_obs["pred"])

# Print evaluation metrics
print(r1, mse)

0.6409841179847717 0.00014962273


In [11]:
# Prepare test input features: convert DataFrame to numpy, add channel dimension, convert to CUDA tensor
test_inputs =  torch.from_numpy(np.expand_dims(df_xy_maxmin.iloc[:,:-1].values,axis=-1)).cuda().float()

# Prepare test output/target values: get last column from DataFrame, reshape, convert to CUDA tensor
test_outputs = torch.from_numpy(df_xy_maxmin.iloc[:,-1].values.reshape(df_xy_maxmin.iloc[:,-1].values.shape[0],1)).cuda().float()

# Create TensorDataset from inputs and outputs
test_tensor = TensorDataset(test_inputs, test_outputs)

# Create DataLoader for batch processing during testing
test_loader = DataLoader(test_tensor, batch_size=100, shuffle = True, drop_last = True)

# Initialize lists to store test predictions and observations
y_test_pred_list=[]
y_test_obs_list=[]
i=0  # Counter for batches (not used in current code)

# Iterate through test data batches
for test_batch_x, test_batch_y in test_loader:
    # Move test input data to device (GPU/CPU)
    test_batch_x=test_batch_x.to(device)
    
    # Denormalize test target values using predefined min/max scaling
    test_batch_y=(test_batch_y*(y1_max-y1_min)+y1_min).to(device)
    
    # Forward pass: get model predictions
    y_test_pred = model(test_batch_x)
    
    # Denormalize predictions to original scale
    y1_test_pred=((y_test_pred)*(y1_max-y1_min)+y1_min)
    
    # Store predictions and observations for this batch
    y_test_pred_list.append(y1_test_pred)
    y_test_obs_list.append(test_batch_y)
    
    # Increment batch counter
    i=i+1

# Concatenate all batch results into single tensors
y_test_pred_np=torch.cat(y_test_pred_list)
y_test_obs_np=torch.cat(y_test_obs_list)

# Import evaluation metrics and libraries
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error
import pandas as pd

# Define file path for saving results
f=folder_path+"/"

# Save predictions and observations as numpy files for future use
np.save(f+"y_xy_pred.npy",y_test_pred_np.cpu().detach().numpy())
np.save(f+"y_xy_obs.npy",y_test_obs_np.cpu().detach().numpy())

# Convert tensors to numpy arrays and reshape for analysis
y_pred=y_test_pred_np.cpu().detach().numpy().reshape(-1,1)[:,0]  # Move to CPU, detach from graph, convert to numpy
y_obs= y_test_obs_np.cpu().detach().numpy().reshape(-1,1)[:,0]   # Move to CPU, detach from graph, convert to numpy

# Create DataFrame for easy comparison and handle missing values
df_pred_obs_xy=pd.DataFrame({"pred":y_pred,"obs":y_obs}).dropna(axis=0,how="any")

# Calculate R-squared (coefficient of determination)
r1=r2_score(df_pred_obs_xy["obs"],df_pred_obs_xy["pred"])

# Calculate Mean Squared Error
mse=mean_squared_error(df_pred_obs_xy["obs"],df_pred_obs_xy["pred"])

# Print evaluation metrics
print(r1, mse)

0.6499574184417725 0.00014361017


# 4. save model

In [13]:
# Save the trained model to a file for future loading and inference
torch.save(model, f+"model.pth")

# Save the test predictions vs observations DataFrame to CSV for analysis
df_pred_obs.to_csv(f+"pred_obs_testing.csv")

# Save the XY dataset predictions vs observations DataFrame to CSV
df_pred_obs_xy.to_csv(f+"pred_obs_xy.csv")

# Save training loss history as numpy array for plotting/analysis
np.save(f+"train_loss.npy", np.array(train_loss_list))

# Save validation loss history as numpy array for plotting/analysis
np.save(f+"val_loss.npy", np.array(val_loss_list))

# Save the shuffled indices used during data splitting for reproducibility
np.save(f+"shuffled_indices.npy", shuffled_indices)